# Quickstart

pyGTFSHandler loads one or more GTFS feeds into a single `Feed` object and
gives you:

- An interactive route/timetable map (`route_map`)
- Stop-level and edge-level **speed** and **headway** analysis
- Polars DataFrames for your own analysis

This notebook uses small local GTFS feeds bundled in the repo
(`examples/test_files/sevilla`), so it runs offline with no downloads.
For a full real-world workflow (downloading feeds, computing service
intensity, exporting GIS layers, etc.) see
`cambridge_massachusetts_usa_example.ipynb`.


In [ ]:
from pathlib import Path
import datetime

from pyGTFSHandler import Feed
from pyGTFSHandler.maps import route_map

OUTPUT_DIR = Path("outputs/quickstart")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("test_files/sevilla")


## 1. Load a feed

- `Feed` accepts one path or a list of paths (one per GTFS zip/folder)
- `stop_group_distance` merges stops within N meters into a shared
  `parent_station` (e.g. platforms on either side of a road)


In [ ]:
feed = Feed(
    [DATA_DIR / "Metro_Sevilla", DATA_DIR / "TUSSAM"],
    stop_group_distance=100,
)
feed


## 2. Interactive route map

`route_map` builds one self-contained Leaflet map for a given service date:

- Every stop is shown as its route-type emoji (bus/rail/subway/...)
- Click a stop to open its timetable
- Click a timetable row to open the full trip itinerary
- **"Filter lines…"** and per-mode checkboxes narrow the map down live
- **"Color by"** -> *Speed* or *Headway* recolors stops and segments


In [ ]:
service_date = datetime.date(2026, 7, 6)  # any date within the feed's calendar

m = route_map(feed, service_date)
m.save(str(OUTPUT_DIR / "map.html"))
m


## 3. Speed and headway as DataFrames

Every `get_*` method returns a Polars DataFrame (or LazyFrame), grouped by
whichever key you pass via `by=` and `at=`.


In [ ]:
start_time = datetime.time(7, 0)
end_time = datetime.time(9, 0)

stop_speed = feed.get_speed_at_stops(
    date=service_date,
    start_time=start_time,
    end_time=end_time,
    by="route_id",      # group individual trip speeds by this column
    at="parent_station", # compute for every 'parent_station', 'stop_id' or 'route_id'
    how="mean",          # 'mean', 'max' or 'min'
)
stop_speed.head()


In [ ]:
stop_headway = feed.get_headway_at_stops(
    date=service_date,
    start_time=start_time,
    end_time=end_time,
    by="shape_direction",  # group trips by geometric direction, no direction_id needed
    at="parent_station",
    how="best",             # 'best', 'mean' or 'all'
    n_divisions=1,           # 1 -> 2 direction bins per stop (outbound/inbound)
)
stop_headway.head()


## Next steps

- `cambridge_massachusetts_usa_example.ipynb` -- a complete real-world
  workflow: downloading feeds, service intensity, GIS exports, and the full
  DataFrame column reference
- `direction_and_headway_methodology.ipynb` -- how `direction_id` and
  headway are actually computed, including the `direction_id`-conflict
  inspector (`conflict_map`), with synthetic and real-feed examples
